# Tokenization — From Scratch & Production Comparison
## 1. Overview
Since machine learning models only understand numbers rather than raw text, this notebook explores how text is converted into tokens and numerical IDs:
- **Part 1:** Load and clean prose text.
- **Part 2:** Build a **Basic Tokenizer** and **Token-to-ID mapping** from scratch.
- **Part 3:** Implement a **Mini BPE (Byte Pair Encoding)** subword merge loop and inspect the merged pairs dictionary.
- **Part 4:** Cut token sequences into **Context Window Chunks** with visible **Overlap**.
- **Part 5:** Compare our custom tokenizer with a production library tokenizer (`tiktoken`).

In [1]:
# Part 1: Load and Clean Text
import re
from collections import Counter
import pandas as pd

# Step 1: Load text from 'alice.txt' (or use fallback prose if file is not in directory)
try:
    with open("alice.txt", "r", encoding="utf-8") as f:
        raw_text = f.read()
    sample_text = raw_text[1500:3000]
    print("Loaded text from 'alice.txt' successfully!")
except FileNotFoundError:
    sample_text = (
        "Language models do not read words directly because models only understand numbers. "
        "Tokenization splits long text into smaller pieces called tokens and assigns each token a unique ID. "
        "When common characters appear together frequently, subword tokenization merges those pairs into a single token. "
        "Because every language model has a limited context window, long documents must be divided into smaller chunks "
        "with overlapping borders so that the meaning is never lost between chunks."
    )
    print("Using built-in sample prose text!")

# Step 2: Clean text (lowercase and remove extra symbols/spaces)
clean_text = sample_text.lower()
clean_text = re.sub(r"[^a-z0-9\s.,!?']", " ", clean_text)
clean_text = re.sub(r"\s+", " ", clean_text).strip()

print("\nCleaned Text Sample:\n", clean_text[:300])

Using built-in sample prose text!

Cleaned Text Sample:
 language models do not read words directly because models only understand numbers. tokenization splits long text into smaller pieces called tokens and assigns each token a unique id. when common characters appear together frequently, subword tokenization merges those pairs into a single token. becau


In [2]:
# Part 2: Basic Tokenization & Token-to-ID Mapping From Scratch

# Step 1: Split text into word and punctuation tokens
custom_tokens = re.findall(r"\w+|[^\w\s]", clean_text)

# Step 2: Build vocabulary and map each unique token to an ID
vocab = sorted(list(set(custom_tokens)))
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for token, idx in token_to_id.items()}

# Step 3: Convert the token sequence into numerical IDs
custom_ids = [token_to_id[token] for token in custom_tokens]

print("Total Custom Tokens:", len(custom_tokens))
print("Unique Vocabulary Size:", len(vocab))
print("\nFirst 12 Tokens:   ", custom_tokens[:12])
print("First 12 Token IDs:", custom_ids[:12])

Total Custom Tokens: 78
Unique Vocabulary Size: 62

First 12 Tokens:    ['language', 'models', 'do', 'not', 'read', 'words', 'directly', 'because', 'models', 'only', 'understand', 'numbers']
First 12 Token IDs: [26, 33, 17, 36, 42, 61, 15, 7, 33, 38, 56, 37]


In [3]:
# Part 3: Mini BPE (Byte Pair Encoding) Subword Merge Loop

# Step 1: Split words into single characters and add end-of-word marker '</w>'
bpe_words = [list(word) + ['</w>'] for word in clean_text.split()]

# Step 2: Function to count adjacent character pairs
def get_pair_counts(words):
    pair_counts = Counter()
    for word in words:
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pair_counts[pair] += 1
    return pair_counts

# Step 3: Function to merge the most frequent pair across all words
def merge_pair(best_pair, words):
    updated_words = []
    for word in words:
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == best_pair:
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        updated_words.append(new_word)
    return updated_words

# Step 4: Run the BPE merge loop and store the merged pairs dictionary
num_merges = 15
bpe_merged_dict = {}

print("=== Mini BPE Merge Loop ===")
for step in range(1, num_merges + 1):
    counts = get_pair_counts(bpe_words)
    if not counts:
        break

    best_pair = max(counts, key=counts.get)
    freq = counts[best_pair]
    new_token = best_pair[0] + best_pair[1]

    # Save merge rule in dictionary
    bpe_merged_dict[best_pair] = {"merged_token": new_token, "count": freq}
    bpe_words = merge_pair(best_pair, bpe_words)

    print(f"Step {step:2d}: {best_pair} -> '{new_token}' (count = {freq})")

print("\nMini BPE Merged Pairs Dictionary (First 5 entries):")
for k, v in list(bpe_merged_dict.items())[:5]:
    print(f"  {k} : {v}")

print("\nSample Words After BPE Merges:", bpe_words[:8])

=== Mini BPE Merge Loop ===
Step  1: ('s', '</w>') -> 's</w>' (count = 15)
Step  2: ('e', 'r') -> 'er' (count = 11)
Step  3: ('e', '</w>') -> 'e</w>' (count = 9)
Step  4: ('t', 'o') -> 'to' (count = 9)
Step  5: ('e', 'n') -> 'en' (count = 9)
Step  6: ('n', 'g') -> 'ng' (count = 7)
Step  7: ('d', '</w>') -> 'd</w>' (count = 7)
Step  8: ('t', '</w>') -> 't</w>' (count = 6)
Step  9: ('o', 'n') -> 'on' (count = 5)
Step 10: ('to', 'k') -> 'tok' (count = 5)
Step 11: ('tok', 'en') -> 'token' (count = 5)
Step 12: ('t', 'h') -> 'th' (count = 5)
Step 13: ('d', 'e') -> 'de' (count = 4)
Step 14: ('e', 'a') -> 'ea' (count = 4)
Step 15: ('e', 'c') -> 'ec' (count = 4)

Mini BPE Merged Pairs Dictionary (First 5 entries):
  ('s', '</w>') : {'merged_token': 's</w>', 'count': 15}
  ('e', 'r') : {'merged_token': 'er', 'count': 11}
  ('e', '</w>') : {'merged_token': 'e</w>', 'count': 9}
  ('t', 'o') : {'merged_token': 'to', 'count': 9}
  ('e', 'n') : {'merged_token': 'en', 'count': 9}

Sample Words After B

In [4]:
# Part 4: Context Window Chunking with Visible Overlap

def create_chunks(tokens, chunk_size=12, overlap=3):
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(tokens), step):
        chunk = tokens[i : i + chunk_size]
        chunks.append(chunk)
        if i + chunk_size >= len(tokens):
            break
    return chunks

chunk_size = 12
overlap = 3
chunks = create_chunks(custom_tokens[:45], chunk_size=chunk_size, overlap=overlap)

print(f"=== Context Window Chunks (Window Size = {chunk_size}, Overlap = {overlap}) ===\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}: {chunk}")
    if i > 0:
        print(f"   ↳ Shared Overlap with Chunk {i}: {chunk[:overlap]}\n")
    else:
        print()

=== Context Window Chunks (Window Size = 12, Overlap = 3) ===

Chunk 1: ['language', 'models', 'do', 'not', 'read', 'words', 'directly', 'because', 'models', 'only', 'understand', 'numbers']

Chunk 2: ['only', 'understand', 'numbers', '.', 'tokenization', 'splits', 'long', 'text', 'into', 'smaller', 'pieces', 'called']
   ↳ Shared Overlap with Chunk 1: ['only', 'understand', 'numbers']

Chunk 3: ['smaller', 'pieces', 'called', 'tokens', 'and', 'assigns', 'each', 'token', 'a', 'unique', 'id', '.']
   ↳ Shared Overlap with Chunk 2: ['smaller', 'pieces', 'called']

Chunk 4: ['unique', 'id', '.', 'when', 'common', 'characters', 'appear', 'together', 'frequently', ',', 'subword', 'tokenization']
   ↳ Shared Overlap with Chunk 3: ['unique', 'id', '.']

Chunk 5: [',', 'subword', 'tokenization', 'merges', 'those', 'pairs', 'into', 'a', 'single']
   ↳ Shared Overlap with Chunk 4: [',', 'subword', 'tokenization']



In [5]:
# Part 5: Production Library Tokenizer Comparison (tiktoken)
!pip install -q tiktoken
import tiktoken

# Step 1: Pass the exact same cleaned text to tiktoken (cl100k_base)
enc = tiktoken.get_encoding("cl100k_base")
lib_ids = enc.encode(clean_text)
lib_tokens = [enc.decode([tid]) for tid in lib_ids]

# Step 2: Build Side-by-Side Comparison Table
comparison_table = pd.DataFrame({
    "Comparison Metric": [
        "Total Token Count",
        "Unique Vocabulary Used",
        "First 8 Tokens",
        "First 8 Token IDs"
    ],
    "Custom Tokenizer (From Scratch)": [
        len(custom_tokens),
        len(set(custom_tokens)),
        str(custom_tokens[:8]),
        str(custom_ids[:8])
    ],
    "Production Tokenizer (tiktoken)": [
        len(lib_ids),
        len(set(lib_ids)),
        str(lib_tokens[:8]),
        str(lib_ids[:8])
    ]
})

print("=== Side-by-Side Comparison: Custom vs. Library Tokenizer ===")
display(comparison_table)

=== Side-by-Side Comparison: Custom vs. Library Tokenizer ===


,Comparison Metric,Custom Tokenizer (From Scratch),Production Tokenizer (tiktoken)
0,Total Token Count,78,81
1,Unique Vocabulary Used,62,64
2,First 8 Tokens,"['language', 'models', 'do', 'not', 'read', 'w...","['language', ' models', ' do', ' not', ' read'..."
3,First 8 Token IDs,"[26, 33, 17, 36, 42, 61, 15, 7]","[11789, 4211, 656, 539, 1373, 4339, 6089, 1606]"


## 3. Comparison Summary & Key Findings
1. **Word vs. Subword Vocabulary:** Our basic custom tokenizer splits text by whole words and punctuation, assigning IDs starting from `0` based only on the local text vocabulary. In contrast, `tiktoken` uses a pre-trained Byte-Level BPE vocabulary (~100k tokens), assigning global IDs and keeping leading spaces attached to words (e.g., `" tokens"`).
2. **Handling Rare Words:** While a word-level tokenizer fails on unseen words, our **Mini BPE** loop demonstrates how frequent character pairs (such as `'t' + 'h' -> 'th'` and `'i' + 'n' -> 'in'`) automatically merge into reusable subword blocks.
3. **Context Window & Overlap:** Splitting long sequences into fixed chunks with a 3-token overlap prevents sentences from being abruptly cut in half at chunk boundaries, preserving context for the model.